# WESAD: Four Baselines with LOSO and Three Seeds

This notebook compares four classifiers on the corrected WESAD wrist-window CSVs: Majority Dummy, Logistic Regression, RBF Support Vector Machine, and Hoeffding Adaptive Tree (HAT) with ADWIN. It uses leave-one-subject-out (LOSO) validation and three seeds.

**Environment:** Colab already provides NumPy, pandas, and scikit-learn. This notebook does not reinstall them. It installs only River, and checks imports before model execution. If the initial NumPy check fails, reset the Colab runtime before proceeding.

**Interpretation:** The first three models are fixed batch baselines evaluated on an entirely held-out participant. HAT is evaluated in prequential replay mode: it predicts each held-out window, then learns from the recorded label. This is online test-time adaptation, and should be reported separately. WESAD contains no interface interaction outcomes, so this notebook evaluates affect classification only.

In [ ]:
# Check the runtime's bundled scientific stack first; do not reinstall NumPy/sklearn here.
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import LeaveOneGroupOut
print('NumPy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('Core imports OK')

In [ ]:
# River is the only extra package required for the online HAT baseline.
%pip -q install river==0.22.0

In [ ]:
import river
from river import preprocessing as river_preprocessing
from river import tree
print('River:', river.__version__, '| imports OK')

## 1. Connect Google Drive and load only corrected participant files

The loader accepts files named like `S2_windows.csv`, `S10_windows.csv`, etc. It excludes the older combined `wesad_eda_windows.csv` automatically.

In [ ]:
from google.colab import drive
from pathlib import Path
import re
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
files = sorted(p for p in DATA_DIR.glob('S*_windows.csv')
               if re.fullmatch(r'S\d+_windows', p.stem))
if len(files) != 15:
    raise ValueError(f'Expected 15 corrected participant CSV files in {DATA_DIR}; found {len(files)}')
df = pd.concat((pd.read_csv(p) for p in files), ignore_index=True)
print(f'Loaded {len(files)} files, {len(df):,} windows, {df.subject.nunique()} participants')
display(df.head())

## 2. Audit the dataset and define predictors

Do not use participant ID, window index, or `label_id` as predictors: those are identifiers, time order, and a numeric duplicate of the target label. The dataset should contain 3,281 rows, 15 participants, and the three classes.

In [ ]:
required = {'subject', 'window', 'label', 'label_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
feature_cols = [c for c in df.columns if c not in required]
assert feature_cols, 'No sensor features found'
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce')
assert df[feature_cols].notna().all().all(), 'Missing feature values: check that only corrected CSVs were loaded'
assert np.isfinite(df[feature_cols].to_numpy(dtype=float)).all(), 'Infinite sensor feature values found'
assert df.duplicated().sum() == 0, 'Duplicate rows: check for mixed or repeated CSVs'
assert df.groupby('label').label_id.nunique().max() == 1, 'Inconsistent label-to-ID mapping'
assert df.groupby('subject').window.apply(lambda s: s.is_monotonic_increasing).all(), 'Windows are not chronologically ordered'
assert len(df) == 3281, f'Expected 3281 rows, got {len(df)}; check the files in Drive'
assert df.subject.nunique() == 15, 'Expected 15 participants'
CLASS_ORDER = ['baseline', 'stress', 'amusement']
assert set(df.label.unique()) == set(CLASS_ORDER), f'Unexpected labels: {sorted(df.label.unique())}'
X = df[feature_cols].astype(float)
y = df['label'].astype(str)
groups = df['subject'].astype(str)
print('Rows:', len(df), '| participants:', df.subject.nunique())
print('Class counts:', y.value_counts().to_dict())
print('Missing feature values:', int(df[feature_cols].isna().sum().sum()))
print('Data checks passed.')

## 3. LOSO experiments with three seeds

LOSO holds out each participant in turn. For batch baselines, imputation and scaling are fit only on the 14 training participants. HAT is first trained from the 14 training participants; their subject order is randomized per seed while their within-subject time order is preserved. On the held-out participant, HAT predicts first and then learns the recorded label. The target labels are available in this offline replay; a live system would need delayed user feedback or another valid label source.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

SEEDS = [17, 42, 2026]
logo = LeaveOneGroupOut()
records = []

def metric_row(actual, predicted):
    return {
        'accuracy': accuracy_score(actual, predicted),
        'macro_f1': f1_score(actual, predicted, labels=CLASS_ORDER, average='macro', zero_division=0),
        'macro_precision': precision_score(actual, predicted, labels=CLASS_ORDER, average='macro', zero_division=0),
        'macro_recall': recall_score(actual, predicted, labels=CLASS_ORDER, average='macro', zero_division=0),
    }

for seed in SEEDS:
    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
        train_subjects = set(groups.iloc[train_idx])
        test_subjects = set(groups.iloc[test_idx])
        assert len(test_subjects) == 1 and train_subjects.isdisjoint(test_subjects)
        held_out = next(iter(test_subjects))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        models = {
            'Majority Dummy': make_pipeline(
                SimpleImputer(strategy='median'),
                DummyClassifier(strategy='prior')
            ),
            'Logistic Regression': make_pipeline(
                SimpleImputer(strategy='median'), StandardScaler(),
                LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed)
            ),
            'RBF SVM': make_pipeline(
                SimpleImputer(strategy='median'), StandardScaler(),
                SVC(kernel='rbf', class_weight='balanced', random_state=seed)
            ),
        }
        for model_name, model in models.items():
            model.fit(X_train, y_train)
            prediction = model.predict(X_test)
            records.append({'seed': seed, 'fold': fold, 'held_out_subject': held_out,
                            'model': model_name, 'n_test_windows': len(test_idx),
                            **metric_row(y_test, prediction)})

        # HAT online learner: warm-start on training participants.
        hat = river_preprocessing.StandardScaler() | tree.HoeffdingAdaptiveTreeClassifier(
            seed=seed, bootstrap_sampling=True
        )
        # Shuffle participant order reproducibly; preserve each participant's chronology.
        train_df = df.iloc[train_idx].copy()
        train_subject_order = np.array(sorted(train_df.subject.unique()))
        np.random.default_rng(seed).shuffle(train_subject_order)
        for subject in train_subject_order:
            subject_rows = train_df[train_df.subject == subject].sort_values('window')
            for row in subject_rows.itertuples(index=False):
                xrow = {col: float(getattr(row, col)) for col in feature_cols}
                hat.learn_one(xrow, str(row.label))

        test_df = df.iloc[test_idx].sort_values('window')
        online_actual, online_prediction = [], []
        for row in test_df.itertuples(index=False):
            xrow = {col: float(getattr(row, col)) for col in feature_cols}
            actual = str(row.label)
            pred = hat.predict_one(xrow)  # predict before the label is revealed
            online_actual.append(actual)
            online_prediction.append(pred if pred is not None else CLASS_ORDER[0])
            hat.learn_one(xrow, actual)  # replay feedback/update after prediction
        records.append({'seed': seed, 'fold': fold, 'held_out_subject': held_out,
                        'model': 'Hoeffding Adaptive Tree (ADWIN)',
                        'n_test_windows': len(test_df),
                        **metric_row(online_actual, online_prediction)})

results = pd.DataFrame(records)
print(f'Completed {len(results)} model × seed × participant evaluations')
display(results.head(12))

## 4. Summarize fairly across 15 participants

The three seeds are repeats, not extra participants. First average scores across seeds within each held-out participant; then report the mean and standard deviation across the 15 participants.

In [ ]:
score_cols = ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall']
per_subject = (results.groupby(['model', 'held_out_subject'], as_index=False)[score_cols]
               .mean())
summary = (per_subject.groupby('model')[score_cols]
           .agg(['mean', 'std']).round(3))
display(summary)
print('Per-seed means (diagnostic):')
display(results.groupby(['model', 'seed'])[score_cols].mean().round(3))
print('Per-participant scores (averaged over seeds):')
display(per_subject.sort_values(['model', 'held_out_subject']))

## 5. Save results to Google Drive

In [ ]:
OUT_DIR = DATA_DIR / 'baseline_results'
OUT_DIR.mkdir(exist_ok=True)
results.to_csv(OUT_DIR / 'wesad_loso_4baselines_3seeds_raw.csv', index=False)
per_subject.to_csv(OUT_DIR / 'wesad_loso_4baselines_per_subject.csv', index=False)
summary.to_csv(OUT_DIR / 'wesad_loso_4baselines_summary.csv')
print('Saved three result files to', OUT_DIR)

## Reporting notes

- Report mean ± standard deviation across held-out participants, and include macro-F1 plus per-class precision/recall.
- Majority Dummy is a reference floor. Logistic Regression and SVM are fixed batch baselines. HAT is a replay-based online adaptation baseline; report separately because it learns from labels on the held-out participant after each prediction.
- Keep LOSO grouping. Do not randomly split windows across train and test.
- Results depend on the validity of sensor-label alignment and the corrected CSV preprocessing. Confirm synchronization against WESAD metadata before publication.
- WESAD supports wearable affect classification experiments; it contains no interface interaction outcomes, so it cannot establish that UI adaptation improves usability.